# Clean-subset evaluation: baseline vs finetuned Llama-3.2-1B (Verilog)
Evaluates both models on the **390 test modules** with no identical copy in training and a testbench that its reference solution passes cleanly (`data/test_ids_dedup.txt`).
Same prompt, sampling (temperature 0.2, 1024 new tokens) and scoring rule as `finetune_qlora_colab.ipynb`.

**Before running:** Runtime -> T4 GPU; add Colab secret `HF_TOKEN` (needs read access to the private repo `arthurjawetz/llama3.2-1b-verilog-qlora` and to Llama-3.2). Upload `eval_bundle.zip` in Cell 1.

In [ ]:
# Cell 1: upload and extract eval_bundle.zip
from google.colab import files
import zipfile, os
up = files.upload()
with zipfile.ZipFile(list(up)[0]) as z: z.extractall('/content/project')
os.chdir('/content/project'); print(os.listdir('.'))

In [ ]:
# Cell 2: dependencies
%%capture
!pip install unsloth
!apt-get install -y -q iverilog

In [ ]:
# Cell 3: HF login
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [ ]:
# Cell 4: helpers (identical to finetune_qlora_colab.ipynb Cell 7 / 9)
import json, re, subprocess, torch
from pathlib import Path
from unsloth import FastLanguageModel

BASE = 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit'
ADAPTER = 'arthurjawetz/llama3.2-1b-verilog-qlora'
MAX_NEW_TOKENS, TEMPERATURE, SEED = 1024, 0.2, 42
IDS = [l.strip() for l in open('data/test_ids_dedup.txt') if l.strip()]
print(len(IDS), 'modules')

def build_prompt(spec):
    module_name = spec.get('module_name', 'unknown')
    description = spec.get('description', '').strip()
    port_lines = []
    for p in spec.get('ports', []):
        w = p.get('bit_width', 1)
        width_str = f'[{w-1}:0] ' if isinstance(w, int) and w > 1 else ''
        port_lines.append(f"  {p['direction']} {width_str}{p['name']}")
    port_str = '\n'.join(port_lines) if port_lines else '  (no ports listed)'
    return (f'Generate a Verilog module named `{module_name}` with the following specification.\n\n'
            f'Description: {description}\n\nPorts:\n{port_str}\n\n'
            'Respond with only the complete Verilog module code. Do not include markdown fences or any explanation.')

def extract_verilog(text):
    if '```' in text:
        lines, out, inside = text.split('\n'), [], False
        for line in lines:
            if line.strip().startswith('```'):
                inside = not inside; continue
            if inside or not any(l.strip().startswith('```') for l in lines): out.append(line)
        text = '\n'.join(out)
    m = re.search(r'\bendmodule\b', text, re.IGNORECASE)
    if m:
        end = text.find('\n', m.end()); text = text[: end if end != -1 else m.end()]
    return text.strip()

def generate_all(model_name, out_dir):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    model, tok = FastLanguageModel.from_pretrained(model_name=model_name, max_seq_length=2048, dtype=None, load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    if tok.pad_token_id is None: tok.pad_token_id = tok.eos_token_id
    torch.manual_seed(SEED)
    for n, i in enumerate(IDS, 1):
        if (out / f'{i}.v').exists(): continue
        spec = json.load(open(f'VerilogDB/{i}.json'))
        r = tok.apply_chat_template([{'role': 'user', 'content': build_prompt(spec)}], add_generation_prompt=True, return_tensors='pt')
        ids = (r.input_ids if hasattr(r, 'input_ids') else r).to('cuda')
        with torch.no_grad():
            o = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=MAX_NEW_TOKENS,
                               do_sample=True, temperature=TEMPERATURE, pad_token_id=tok.pad_token_id)
        (out / f'{i}.v').write_text(extract_verilog(tok.decode(o[0, ids.shape[1]:], skip_special_tokens=True)))
        if n % 50 == 0: print(f'{model_name}: {n}/{len(IDS)}')
    del model; torch.cuda.empty_cache()

FAIL = re.compile(r'\b(Error:|Failed|FAIL)\b', re.IGNORECASE)
def run_test(v, tb, tag):
    Path('sim_build').mkdir(exist_ok=True); sim = f'sim_build/{tag}'
    try: r = subprocess.run(['iverilog', '-o', sim, str(v), str(tb)], capture_output=True, text=True, timeout=15)
    except subprocess.TimeoutExpired: return 'compile_error'
    if r.returncode: return 'compile_error'
    try: r = subprocess.run(['vvp', sim], capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired: return 'timeout'
    if FAIL.search(r.stdout or '') or FAIL.search(r.stderr or ''): return 'wrong_output'
    return 'pass' if 'PASS' in (r.stdout or '') else 'no_output'

def evaluate(vdir):
    res = {i: run_test(Path(vdir) / f'{i}.v', f'generated_tests/{i}_tb.v', i) for i in IDS}
    n = sum(v == 'pass' for v in res.values())
    return {'passed': n, 'total': len(IDS), 'pass_at_1': round(n / len(IDS), 4), 'per_module': res,
            'categories': {k: sum(v == k for v in res.values()) for k in sorted(set(res.values()))}}

In [ ]:
# Cell 5: generate (baseline, then finetuned adapter from the Hub). ~10-20 min each on a T4.
generate_all(BASE, 'gen/baseline')
generate_all(ADAPTER, 'gen/finetuned')

In [ ]:
# Cell 6: score and compare
b, f = evaluate('gen/baseline'), evaluate('gen/finetuned')
for name, r in (('baseline', b), ('finetuned', f)):
    print(f"{name:10s} pass@1 = {r['pass_at_1']:.3f} ({r['passed']}/{r['total']})  {r['categories']}")
bp, fp = b['per_module'], f['per_module']
print('finetuned gains:', sum(bp[i] != 'pass' and fp[i] == 'pass' for i in IDS), '| loses:', sum(bp[i] == 'pass' and fp[i] != 'pass' for i in IDS))
json.dump({'subset': 'data/test_ids_dedup.txt', 'baseline': b, 'finetuned': f}, open('clean_subset_results.json', 'w'), indent=1)

In [ ]:
# Cell 7: download
from google.colab import files
files.download('clean_subset_results.json')